# 02. Limpieza de datos

**Objetivo**: dejar las 7 tablas en un estado en el que se pueda confiar en ellas — fechas y textos estandarizados, duplicados resueltos, nulos decididos (rellenados, documentados como estructurales, o dejados como están con justificación) — sin calcular todavía ninguna métrica.

In [1]:
# Cargamos las librerias
import pandas as pd

In [2]:
# Lista con los nombres base de los 7 archivos 
ficheros = ['usuarios','sesiones','partidas','compras','ingresos_ads','gasto_ua','productos_iap']

# Carpeta donde están los CSV, 
ruta = '../data/raw/'

# Diccionario vacio para guardar los DataFrames
dfs = {}

# Bucle para recorrer la lista y construir la ruta completa
for f in ficheros:
    dfs[f] = pd.read_csv(f"{ruta}{f}.csv")
    # Imprimimos el nombre del fichero y su forma (filas, columnas
    print(f"{f}: {dfs[f].shape}")


usuarios: (8040, 6)
sesiones: (44945, 5)
partidas: (61716, 7)
compras: (4558, 8)
ingresos_ads: (19999, 6)
gasto_ua: (280, 6)
productos_iap: (14, 5)


### Limpieza usuarios

In [3]:
# Asignamos el dataframe a una variable
usuarios_limpio = dfs['usuarios'].copy()

print(usuarios_limpio.dtypes)

user_id              int64
fecha_instalacion      str
canal_adquisicion      str
pais                   str
plataforma             str
version_app            str
dtype: object


In [4]:
# Empezamos con la fecha, habia varios formatos 
print(usuarios_limpio['fecha_instalacion'].unique())

<StringArray>
[ '9 jul 2024',  '12/07/2024',  '2024-06-19',  '2024-07-03',  '2024-07-01',
  '03/07/2024',  '28/06/2024',  '2024-07-04',  '15/06/2024',  '09/06/2024',
 ...
 '12 jul 2024',  '2024-05-31', '28 may 2024', '11 jul 2024', '20 jun 2024',
 '29 jun 2024',  '5 jul 2024', '29 may 2024', '11 jun 2024', '13 jun 2024']
Length: 165, dtype: str


In [5]:
# Cambiamos a formato fecha los diferrentes tipos 
usuarios_limpio['fecha_instalacion'] = pd.to_datetime(usuarios_limpio['fecha_instalacion'],
                                                       format='mixed', dayfirst=True, errors='coerce')

In [6]:
# Al poner coerce si algun valor no se interpreta el programa lo convierte en NaT, nos aseguramos de que no genere nulos
print(f"Fechas no reconocidas (NaT): {usuarios_limpio['fecha_instalacion'].isnull().sum()}")
print(f"Primera instalacion: {usuarios_limpio['fecha_instalacion'].min().date()}")
print(f"Ultima instalacion: {usuarios_limpio['fecha_instalacion'].max().date()}")

Fechas no reconocidas (NaT): 0
Primera instalacion: 2024-01-06
Ultima instalacion: 2024-12-07


In [7]:
# Comprobamos los diferentes valores de canal_adquisicion
print(f"Grafias antes: {usuarios_limpio['canal_adquisicion'].nunique()}")
for canal,n in usuarios_limpio['canal_adquisicion'].value_counts().items():
    print(f"{canal}: {n}")

Grafias antes: 23
coinrush: 1281
CoinRush: 855
meta: 636
google: 578
Meta: 409
coin_rush: 407
Google Ads: 399
facebook: 397
tiktok: 355
apple_search: 311
google_ads: 279
ASA: 279
Coinrush: 270
organico: 266
TikTok: 223
Apple Search: 179
Facebook: 168
orgánico: 159
Google: 151
apple search: 137
organic: 132
tik_tok: 124
Organico: 45


In [8]:
# Quitamos mayuculas, espacios y mapeamos para formatear los diferentes valors
mapa = {
    'coinrush': 'coinrush', 'coin_rush': 'coinrush',
    'meta': 'meta', 'facebook': 'meta',
    'google': 'google', 'google ads': 'google', 'google_ads': 'google',
    'apple_search': 'apple_search', 'asa': 'apple_search', 'apple search': 'apple_search',
    'tiktok': 'tiktok', 'tik_tok': 'tiktok',
    'organico': 'organico', 'orgánico': 'organico', 'organic': 'organico',
}

usuarios_limpio['canal_adquisicion'] =  usuarios_limpio['canal_adquisicion'].str.lower().str.strip().map(mapa)

print(f"Grafias despues: {usuarios_limpio['canal_adquisicion'].nunique()}")
for canal,n in usuarios_limpio['canal_adquisicion'].value_counts().items():
    print(f"{canal}: {n}")

Grafias despues: 6
coinrush: 2813
meta: 1610
google: 1407
apple_search: 906
tiktok: 702
organico: 602


In [9]:
# Pasamos al pais, comprobamos primero los valores para sacar el mapa 
print(f"Grafias antes: {usuarios_limpio['pais'].nunique()}")
for pais,n in usuarios_limpio['pais'].value_counts().items():
    print(f"{pais}: {n}")

Grafias antes: 12
ES: 1864
MX: 1268
España: 834
PT: 763
es: 680
IT: 568
México: 554
mexico: 459
Portugal: 348
pt: 292
Italia: 244
it: 166


In [10]:
# Estandarizamos formatos
mapa_pais = {
    'es': 'ES', 'españa': 'ES',
    'mx': 'MX', 'méxico': 'MX', 'mexico': 'MX',
    'pt': 'PT', 'portugal': 'PT',
    'it': 'IT', 'italia': 'IT',
}
usuarios_limpio['pais'] = usuarios_limpio['pais'].str.lower().str.strip().map(mapa_pais)

# Comprobación de nulos
print(f"Nulso tras mapear pais: {usuarios_limpio['pais'].isnull().sum()}")

print(f"Grafias despues: {usuarios_limpio['pais'].nunique()}")
for pais, n in usuarios_limpio['pais'].value_counts().items():
    print(f"{pais}: {n}")

Nulso tras mapear pais: 0
Grafias despues: 4
ES: 3378
MX: 2281
PT: 1403
IT: 978


In [11]:
# Pasamos a la plataforma:
print(f"Grafias antes: {usuarios_limpio['plataforma'].nunique()}")
for plataforma, n in usuarios_limpio['plataforma'].value_counts().items():
    print(f"{plataforma}: {n}")

Grafias antes: 6
android: 1995
Android: 1747
ios: 1641
iOS: 1485
AND: 629
IOS: 543


In [12]:
# Transformamos estandarizando
mapa_plataforma = {
    'android': 'Android',
    'and': 'Android',
    'ios': 'iOS',
}

usuarios_limpio['plataforma'] = usuarios_limpio['plataforma'].str.lower().str.strip().map(mapa_plataforma)

print(f"Nulos tras la estandarizacion: {usuarios_limpio['plataforma'].isnull().sum()}")
print(f"Grafias despues: {usuarios_limpio['plataforma'].nunique()}")
for plataforma, n in usuarios_limpio['plataforma'].value_counts().items():
    print(f"{plataforma}: {n}")


Nulos tras la estandarizacion: 0
Grafias despues: 2
Android: 4371
iOS: 3669


In [13]:
# Duplicados de user_id
antes = len(usuarios_limpio)

usuarios_limpio = usuarios_limpio.sort_values('fecha_instalacion').drop_duplicates(subset='user_id',keep='last')

print(f"Filas antes:{antes}")
print(f"Filas despues: {len(usuarios_limpio)}")

Filas antes:8040
Filas despues: 8000


### Limpieza — usuarios.csv

- `fecha_instalacion` → convertida a `datetime` (0 nulos).
- `canal_adquisicion` → estandarizada, 6 canales reales.
- `pais` → estandarizada, 4 países (`ES`, `MX`, `PT`, `IT`).
- `plataforma` → estandarizada (`Android`, `iOS`).
- **Duplicados**: `drop_duplicates()` resolvió los 40 casos sin pérdida de información real (8.040 → 8.000 filas).
- `version_app`: 297 nulos, se dejan como están (no interviene en las métricas del encuadre).

### Limpieza sesiones


In [14]:
sesiones_limpio = dfs['sesiones'].copy()

sesiones_limpio.dtypes

evento_id            int64
user_id              int64
fecha                  str
duracion_segundos    int64
plataforma             str
dtype: object

In [15]:
# Transformamos la fecha a un unico formato DD-MM-YYY
sesiones_limpio['fecha'] = pd.to_datetime(sesiones_limpio['fecha'],format='mixed',dayfirst=True, errors='coerce')

print(f"Fechas nulas: {sesiones_limpio['fecha'].isnull().sum()}")

Fechas nulas: 0


In [16]:
# Pasamos a la plataforma
print(f"Grafias antes: {sesiones_limpio['plataforma'].nunique()}")
for plataforma, n in sesiones_limpio['plataforma'].value_counts().items():
    print(f"{plataforma}: {n}")

Grafias antes: 6
android: 10765
Android: 9751
ios: 9451
iOS: 8287
AND: 3627
IOS: 3064


In [17]:
# Estandarizamos las plataformas, es igual que en usuarios
sesiones_limpio['plataforma'] =  sesiones_limpio['plataforma'].str.lower().str.strip().map(mapa_plataforma)

print(f"Nulos tras la estandarizacion: {sesiones_limpio['plataforma'].isnull().sum()}")
print(f"Grafias despues: {sesiones_limpio['plataforma'].nunique()}")

Nulos tras la estandarizacion: 0
Grafias despues: 2


In [18]:
# Seguimos con las sesiones, de duraciones imposibles 
sesiones_antes = len(sesiones_limpio)
print(sesiones_limpio['duracion_segundos'].describe())

malas = ((sesiones_limpio['duracion_segundos'] <= 0) | (sesiones_limpio['duracion_segundos'] > 3600))
 
print("\nSesiones antes: ",sesiones_antes)
print(f"Sesiones fuera de rango: {malas.sum()}")
print(f"Duracion media SIN limpiar: {sesiones_limpio['duracion_segundos'].mean():.1f} s")


count     44945.000000
mean        583.905039
std       20005.028244
min         -30.000000
25%         104.000000
50%         180.000000
75%         255.000000
max      999999.000000
Name: duracion_segundos, dtype: float64

Sesiones antes:  44945
Sesiones fuera de rango: 60
Duracion media SIN limpiar: 583.9 s


In [19]:
sesiones_limpio = sesiones_limpio[sesiones_limpio['duracion_segundos'].between(1, 3600)]

print("\nSesiones despues: ",len(sesiones_limpio))
print(f"Duracion media limpia: {sesiones_limpio['duracion_segundos'].mean():.1f} s")



Sesiones despues:  44885
Duracion media limpia: 183.7 s


In [20]:
# Eliminamos los 312 duplicados que habia exactos
print(f"Duplicados exactos: {sesiones_limpio.duplicated().sum()}")

sesiones_limpio = sesiones_limpio.drop_duplicates()

print(f"Filas despues: {len(sesiones_limpio)}")
print(f"Duplicados ahora: {sesiones_limpio.duplicated().sum()}")

Duplicados exactos: 312
Filas despues: 44573
Duplicados ahora: 0


### Limpieza de `sesiones.csv`

- **Filas originales:** 44,945
- **`fecha`**: convertida a datetime (formatos mixtos, sin nulos resultantes)
- **`plataforma`**: estandarizada a `Android` / `iOS` (6 variantes de texto unificadas)
- **`duracion_segundos`**: eliminadas 60 filas fuera de rango (negativos + valor sentinela 999999), aplicando criterio de negocio (máximo 1 hora = 3600s). Media pasó de 583.9s (deformada por outliers) a 183.6s, coherente con la mediana (180s).
- **Duplicados exactos**: eliminadas 312 filas (evento_id repetido con fila idéntica)
- **Filas finales:** 44,573